In [2]:
import os

# Create the folders if they don't already exist
os.makedirs(r'C:\Users\Janhavi\ChurnProject\app\templates', exist_ok=True)
print("✅ Folders created!")


✅ Folders created!


In [4]:
%%writefile C:\Users\Janhavi\ChurnProject\app\main.py

from fastapi import FastAPI, File, UploadFile, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
import pandas as pd
import pickle
import io

app = FastAPI(title="Customer Churn Predictor")
templates = Jinja2Templates(directory="app/templates")

with open('models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('models/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

print("✅ Model loaded!")

@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    df = pd.read_csv(io.BytesIO(contents))

    nominal_cols = ['gender', 'Partner', 'Dependents',
                    'InternetService', 'Contract',
                    'PaymentMethod', 'PaperlessBilling']

    cols_to_encode = [c for c in nominal_cols if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    df = df[feature_columns]
    df_scaled = scaler.transform(df)

    predictions = model.predict(df_scaled)
    probabilities = model.predict_proba(df_scaled)[:, 1]

    results = []
    for i in range(len(predictions)):
        prob = round(float(probabilities[i]) * 100, 1)
        if probabilities[i] > 0.7:
            risk = "🔴 HIGH"
        elif probabilities[i] > 0.4:
            risk = "🟡 MEDIUM"
        else:
            risk = "🟢 LOW"

        results.append({
            "customer": i + 1,
            "churn": "⚠️ Will Churn" if predictions[i] == 1 else "✅ Will Stay",
            "probability": f"{prob}%",
            "risk": risk
        })

    return {"predictions": results}


Overwriting C:\Users\Janhavi\ChurnProject\app\main.py


In [6]:
%%writefile C:\Users\Janhavi\ChurnProject\app\templates\index.html

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Customer Churn Predictor</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Inter', sans-serif;
            background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
        }
        .container {
            background: rgba(255,255,255,0.05);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255,255,255,0.1);
            border-radius: 20px;
            padding: 40px;
            width: 100%;
            max-width: 800px;
            color: white;
        }
        h1 {
            font-size: 2rem;
            font-weight: 700;
            margin-bottom: 8px;
            background: linear-gradient(90deg, #a78bfa, #60a5fa);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }
        .subtitle { color: rgba(255,255,255,0.6); margin-bottom: 30px; font-size: 0.95rem; }
        .upload-area {
            border: 2px dashed rgba(167,139,250,0.5);
            border-radius: 12px;
            padding: 30px;
            text-align: center;
            margin-bottom: 20px;
            transition: all 0.3s;
        }
        .upload-area:hover { border-color: #a78bfa; background: rgba(167,139,250,0.05); }
        input[type="file"] { display: none; }
        .upload-label { cursor: pointer; display: block; }
        .upload-icon { font-size: 2.5rem; margin-bottom: 10px; }
        .file-name { color: #a78bfa; font-size: 0.9rem; margin-top: 8px; }
        button {
            width: 100%;
            padding: 14px;
            background: linear-gradient(90deg, #7c3aed, #2563eb);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 1rem;
            font-weight: 600;
            cursor: pointer;
            transition: opacity 0.2s;
        }
        button:hover { opacity: 0.9; }
        button:disabled { opacity: 0.5; cursor: not-allowed; }
        .results { margin-top: 30px; }
        .results h2 { font-size: 1.2rem; margin-bottom: 15px; color: rgba(255,255,255,0.8); }
        table { width: 100%; border-collapse: collapse; }
        th {
            background: rgba(124,58,237,0.3);
            padding: 12px;
            text-align: left;
            font-size: 0.85rem;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }
        td { padding: 12px; border-bottom: 1px solid rgba(255,255,255,0.05); font-size: 0.9rem; }
        tr:hover td { background: rgba(255,255,255,0.03); }
        .loader { display: none; text-align: center; padding: 20px; color: rgba(255,255,255,0.6); }
        .info-box {
            background: rgba(96,165,250,0.1);
            border: 1px solid rgba(96,165,250,0.3);
            border-radius: 10px;
            padding: 15px;
            margin-bottom: 25px;
            font-size: 0.85rem;
            color: rgba(255,255,255,0.7);
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🔮 Churn Predictor</h1>
        <p class="subtitle">Upload customer data to identify who is at risk of leaving</p>
        <div class="info-box">
            📋 Upload a CSV file with the same columns as the Telco dataset.
            The model will predict churn probability for each customer.
        </div>
        <div class="upload-area">
            <label class="upload-label" for="csvFile">
                <div class="upload-icon">📂</div>
                <strong>Click to upload CSV</strong>
                <p style="color:rgba(255,255,255,0.5); font-size:0.85rem; margin-top:5px;">Supports .csv files</p>
                <div class="file-name" id="fileName">No file chosen</div>
            </label>
            <input type="file" id="csvFile" accept=".csv">
        </div>
        <button id="predictBtn" onclick="predict()">Predict Churn</button>
        <div class="loader" id="loader">⏳ Analysing customers...</div>
        <div class="results" id="results" style="display:none;">
            <h2>📊 Prediction Results</h2>
            <table id="resultsTable">
                <thead>
                    <tr><th>#</th><th>Prediction</th><th>Probability</th><th>Risk Level</th></tr>
                </thead>
                <tbody id="resultsBody"></tbody>
            </table>
        </div>
    </div>
    <script>
        document.getElementById('csvFile').addEventListener('change', function() {
            const name = this.files[0] ? this.files[0].name : 'No file chosen';
            document.getElementById('fileName').textContent = name;
        });

        async function predict() {
            const file = document.getElementById('csvFile').files[0];
            if (!file) { alert('Please select a CSV file first!'); return; }
            document.getElementById('predictBtn').disabled = true;
            document.getElementById('loader').style.display = 'block';
            document.getElementById('results').style.display = 'none';
            const formData = new FormData();
            formData.append('file', file);
            try {
                const response = await fetch('/predict', { method: 'POST', body: formData });
                const data = await response.json();
                let rows = '';
                data.predictions.forEach(row => {
                    rows += `<tr>
                        <td>${row.customer}</td>
                        <td>${row.churn}</td>
                        <td>${row.probability}</td>
                        <td>${row.risk}</td>
                    </tr>`;
                });
                document.getElementById('resultsBody').innerHTML = rows;
                document.getElementById('results').style.display = 'block';
            } catch (error) {
                alert('Something went wrong. Check Anaconda Prompt for errors.');
            }
            document.getElementById('predictBtn').disabled = false;
            document.getElementById('loader').style.display = 'none';
        }
    </script>
</body>
</html>


Overwriting C:\Users\Janhavi\ChurnProject\app\templates\index.html


In [8]:
import pandas as pd

test_df = pd.read_csv(r'C:\Users\Janhavi\ChurnProject\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv')
test_sample = test_df.drop(['Churn', 'customerID'], axis=1).head(10)
test_sample.to_csv(r'C:\Users\Janhavi\ChurnProject\data\test_upload.csv', index=False)
print("✅ Test file created!")


✅ Test file created!


In [10]:
import os

files_to_check = [
    r'C:\Users\Janhavi\ChurnProject\app\main.py',
    r'C:\Users\Janhavi\ChurnProject\app\templates\index.html',
    r'C:\Users\Janhavi\ChurnProject\models\best_model.pkl',
    r'C:\Users\Janhavi\ChurnProject\models\scaler.pkl',
    r'C:\Users\Janhavi\ChurnProject\models\feature_columns.pkl',
    r'C:\Users\Janhavi\ChurnProject\data\test_upload.csv',
]

for f in files_to_check:
    status = "✅ EXISTS" if os.path.exists(f) else "❌ MISSING"
    print(f"{status} — {f.split(chr(92))[-1]}")


✅ EXISTS — main.py
✅ EXISTS — index.html
✅ EXISTS — best_model.pkl
✅ EXISTS — scaler.pkl
✅ EXISTS — feature_columns.pkl
✅ EXISTS — test_upload.csv


In [12]:
%%writefile C:\Users\Janhavi\ChurnProject\app\main.py

from fastapi import FastAPI, File, UploadFile, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
import pandas as pd
import pickle
import io

app = FastAPI(title="Customer Churn Predictor")
templates = Jinja2Templates(directory="app/templates")

with open('models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('models/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

print("✅ Model loaded!")

@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    # ← THIS LINE IS THE FIX (request= as keyword argument)
    return templates.TemplateResponse(request=request, name="index.html")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    df = pd.read_csv(io.BytesIO(contents))

    nominal_cols = ['gender', 'Partner', 'Dependents',
                    'InternetService', 'Contract',
                    'PaymentMethod', 'PaperlessBilling']

    cols_to_encode = [c for c in nominal_cols if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    df = df[feature_columns]
    df_scaled = scaler.transform(df)

    predictions = model.predict(df_scaled)
    probabilities = model.predict_proba(df_scaled)[:, 1]

    results = []
    for i in range(len(predictions)):
        prob = round(float(probabilities[i]) * 100, 1)
        if probabilities[i] > 0.7:
            risk = "🔴 HIGH"
        elif probabilities[i] > 0.4:
            risk = "🟡 MEDIUM"
        else:
            risk = "🟢 LOW"

        results.append({
            "customer": i + 1,
            "churn": "⚠️ Will Churn" if predictions[i] == 1 else "✅ Will Stay",
            "probability": f"{prob}%",
            "risk": risk
        })

    return {"predictions": results}


Overwriting C:\Users\Janhavi\ChurnProject\app\main.py


In [14]:
%%writefile C:\Users\Janhavi\ChurnProject\app\main.py

from fastapi import FastAPI, File, UploadFile, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
import pandas as pd
import numpy as np
import pickle
import io

app = FastAPI(title="Customer Churn Predictor")
templates = Jinja2Templates(directory="app/templates")

with open('models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('models/feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

print("✅ Model loaded!")

def preprocess(df):
    # ── Step 1: Fix TotalCharges if it's text ──────────────────────────────
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

    # ── Step 2: Convert Yes/No service columns to 1/0 ──────────────────────
    # These are the service columns that had Yes/No/No internet service values
    service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity',
                    'OnlineBackup', 'DeviceProtection', 'TechSupport',
                    'StreamingTV', 'StreamingMovies']

    for col in service_cols:
        if col in df.columns:
            df[col] = df[col].map({
                'Yes': 1, 'No': 0,
                'No internet service': 0,
                'No phone service': 0
            }).fillna(0).astype(int)

    # ── Step 3: Calculate Service_Count (sum of all service columns) ────────
    existing_service_cols = [c for c in service_cols if c in df.columns]
    df['Service_Count'] = df[existing_service_cols].sum(axis=1)

    # ── Step 4: Calculate RFM Scores (same logic as notebook) ──────────────
    # R_Score: lower tenure = higher risk score (4 = newest, 1 = oldest)
    df['R_Score'] = pd.qcut(df['tenure'], q=4,
                             labels=[4, 3, 2, 1],
                             duplicates='drop').astype(int)

    # F_Score: more services = higher score
    df['F_Score'] = pd.qcut(df['Service_Count'], q=4,
                             labels=[1, 2, 3, 4],
                             duplicates='drop').astype(int)

    # M_Score: higher charges = higher score
    df['M_Score'] = pd.qcut(df['TotalCharges'], q=4,
                             labels=[1, 2, 3, 4],
                             duplicates='drop').astype(int)

    # Combined RFM Score
    df['RFM_Score'] = df['R_Score'] + df['F_Score'] + df['M_Score']

    # Segment label
    def assign_segment(score):
        if score >= 9:
            return 'VIP'
        elif score >= 6:
            return 'Loyal'
        elif score >= 4:
            return 'At-Risk'
        else:
            return 'About to Churn'

    df['Segment'] = df['RFM_Score'].apply(assign_segment)

    # ── Step 5: One-Hot Encode nominal columns ──────────────────────────────
    nominal_cols = ['gender', 'Partner', 'Dependents',
                    'InternetService', 'Contract',
                    'PaymentMethod', 'PaperlessBilling', 'Segment']

    cols_to_encode = [c for c in nominal_cols if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

    # ── Step 6: Add missing columns with 0, remove extra columns ───────────
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0

    # Keep only the exact columns the model was trained on, in the right order
    df = df[feature_columns]

    return df


@app.get("/", response_class=HTMLResponse)
async def home(request: Request):
    return templates.TemplateResponse(request=request, name="index.html")


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    df = pd.read_csv(io.BytesIO(contents))

    # Drop columns that won't be needed
    df.drop(columns=['customerID', 'Churn'], errors='ignore', inplace=True)

    # Run full preprocessing
    df_processed = preprocess(df)

    # Scale
    df_scaled = scaler.transform(df_processed)

    # Predict
    predictions = model.predict(df_scaled)
    probabilities = model.predict_proba(df_scaled)[:, 1]

    results = []
    for i in range(len(predictions)):
        prob = round(float(probabilities[i]) * 100, 1)
        if probabilities[i] > 0.7:
            risk = "🔴 HIGH"
        elif probabilities[i] > 0.4:
            risk = "🟡 MEDIUM"
        else:
            risk = "🟢 LOW"

        results.append({
            "customer": i + 1,
            "churn": "⚠️ Will Churn" if predictions[i] == 1 else "✅ Will Stay",
            "probability": f"{prob}%",
            "risk": risk
        })

    return {"predictions": results}


Overwriting C:\Users\Janhavi\ChurnProject\app\main.py
